# Spark Cluster Test Notebook

This notebook tests the connection to the Spark cluster and MinIO (S3).

## 1. Create Spark Session

In [1]:
# Initialize findspark to locate Spark installation
import findspark
findspark.init()

from pyspark.sql import SparkSession
import os

# Create Spark session - configuration is loaded from spark-defaults.conf
# which includes: master URL, S3/MinIO settings, Hive metastore, Delta Lake
spark = SparkSession.builder \
    .appName("Spark-Cluster-Test") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"\nSpark UI: http://localhost:4040")
print(f"Spark Master UI: http://localhost:8085")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 10:31:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0
Spark master: spark://spark-master:7077
Application ID: app-20260129103128-0000

Spark UI: http://localhost:4040
Spark Master UI: http://localhost:8085


## 2. Test MinIO (S3) Connection

In [2]:
# Create test DataFrame
test_data = [
    (1, "Alice", 100.0),
    (2, "Bob", 200.0),
    (3, "Charlie", 300.0),
    (4, "Diana", 400.0),
    (5, "Eve", 500.0)
]

df = spark.createDataFrame(test_data, ["id", "name", "amount"])
print("Test DataFrame created:")
df.show()

26/01/29 10:31:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Test DataFrame created:


+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|  Alice| 100.0|
|  2|    Bob| 200.0|
|  3|Charlie| 300.0|
|  4|  Diana| 400.0|
|  5|    Eve| 500.0|
+---+-------+------+



In [3]:
# Write to MinIO as Parquet
output_path = "s3a://datalake/spark-test/test-data/"

df.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Data written to: {output_path}")

Data written to: s3a://datalake/spark-test/test-data/


In [4]:
# Read back from MinIO
df_read = spark.read.parquet(output_path)

print(f"Data read from: {output_path}")
print(f"Row count: {df_read.count()}")
df_read.show()

Data read from: s3a://datalake/spark-test/test-data/
Row count: 5
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  3|Charlie| 300.0|
|  1|  Alice| 100.0|
|  4|  Diana| 400.0|
|  2|    Bob| 200.0|
|  5|    Eve| 500.0|
+---+-------+------+



## 3. Test Hive Metastore Connection

In [5]:
# Show databases (should connect to Hive Metastore)
print("Databases:")
spark.sql("SHOW DATABASES").show()

print("\nTables in default database:")
spark.sql("SHOW TABLES IN default").show()

Databases:
+---------+
|namespace|
+---------+
| datalake|
|  default|
+---------+


Tables in default database:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



## 4. Test Delta Lake

In [6]:
# Write as Delta table
delta_path = "s3a://datalake/spark-test/delta-test/"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print(f"Delta table written to: {delta_path}")

26/01/29 10:31:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta table written to: s3a://datalake/spark-test/delta-test/


In [7]:
# Read Delta table
df_delta = spark.read.format("delta").load(delta_path)

print("Delta table content:")
df_delta.show()

Delta table content:


+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  3|Charlie| 300.0|
|  1|  Alice| 100.0|
|  4|  Diana| 400.0|
|  2|    Bob| 200.0|
|  5|    Eve| 500.0|
+---+-------+------+



In [8]:
# Delta table history (time travel)
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

print("Delta table history:")
delta_table.history().show(truncate=False)

Delta table history:
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|2      |2026-01-29 10:32:05|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |1          |Serializable  |false        |{numFiles -> 

## 5. Test SQL Operations

In [9]:
# Register temp view and run SQL
df.createOrReplaceTempView("test_data")

result = spark.sql("""
    SELECT 
        name,
        amount,
        amount * 1.1 as amount_with_tax
    FROM test_data
    WHERE amount > 200
    ORDER BY amount DESC
""")

print("SQL query result:")
result.show()

SQL query result:
+-------+------+------------------+
|   name|amount|   amount_with_tax|
+-------+------+------------------+
|    Eve| 500.0|             550.0|
|  Diana| 400.0|440.00000000000006|
|Charlie| 300.0|             330.0|
+-------+------+------------------+



## 6. Check Cluster Status

In [10]:
# Get cluster information
sc = spark.sparkContext

print(f"Application Name: {sc.appName}")
print(f"Master: {sc.master}")
print(f"Default Parallelism: {sc.defaultParallelism}")
print(f"\nExecutors (check Spark UI for details): http://localhost:4040/executors")

Application Name: Spark-Cluster-Test
Master: spark://spark-master:7077
Default Parallelism: 12

Executors (check Spark UI for details): http://localhost:4040/executors


## 7. Cleanup

In [11]:
# Stop Spark session when done
# Uncomment the line below to stop the session
# spark.stop()

print("Test completed successfully!")
print("\nTo stop the Spark session, run: spark.stop()")

Test completed successfully!

To stop the Spark session, run: spark.stop()


## 8. Reading Existing Data (if available)

In [12]:
# Try to read existing order-events data (if Kafka Connect has written data)
try:
    df_orders = spark.read.parquet("s3a://datalake/topics/order-events/")
    print(f"Found order-events data!")
    print(f"Total rows: {df_orders.count()}")
    print(f"\nSchema:")
    df_orders.printSchema()
    print(f"\nSample data:")
    df_orders.show(5)
except Exception as e:
    print(f"No order-events data found (this is normal if Kafka Connect hasn't written data yet)")
    print(f"Error: {e}")

Found order-events data!


Total rows: 2000000

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- shipping_address: string (nullable = true)
 |-- billing_address: string (nullable = true)
 |-- shipping_zip: string (nullable = true)
 |-- billing_zip: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- billing_city: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- billing_country: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- card_last_digits: string (nullable = true)
 |-- card_expiry: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-